# Diffusion Rotation on Colored Shapes

This notebook loads an open-source diffusion model (InstructPix2Pix) and probes whether it can **rotate** a synthetic colored-shapes image by a specified angle.

We generate a simple colored-rectangles dataset, provide the input image plus an instruction like *"Rotate by 45 degrees clockwise"*, and compare the model output to a ground-truth geometric rotation.


In [ ]:
# %pip -q install -U "diffusers==0.37.0" \
#     "huggingface_hub>=0.24.0" \
#     "transformers>=4.40.0" \
#     "accelerate>=0.27.2" \
#     "safetensors>=0.4.3" \
#     pillow


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import math
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler

plt.rcParams.update({"figure.dpi": 130})

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f'Using device: {device}')


Using device: mps


In [4]:
# --- Colored shapes dataset (synthetic) ---
def random_colored_rectangles(h, w, num_shapes, rng):
    img = np.zeros((h, w, 3), dtype=np.float32)
    for _ in range(num_shapes):
        color = rng.random(3).astype(np.float32)
        y0 = rng.integers(0, h - 8)
        x0 = rng.integers(0, w - 8)
        y1 = min(h, y0 + rng.integers(6, h // 2))
        x1 = min(w, x0 + rng.integers(6, w // 2))
        img[y0:y1, x0:x1] = color
    return img

def rotate_image_np(img, angle_deg):
    # PIL rotates counter-clockwise for positive angles.
    pil = Image.fromarray((img * 255).astype(np.uint8))
    rot = pil.rotate(angle_deg, resample=Image.BILINEAR, expand=False, fillcolor=(0, 0, 0))
    return np.asarray(rot).astype(np.float32) / 255.0

def make_sample_batch(n=4, size=128, num_shapes=4, seed=0):
    rng = np.random.default_rng(seed)
    return [random_colored_rectangles(size, size, num_shapes, rng) for _ in range(n)]

In [ ]:
# --- Load open-source diffusion model (InstructPix2Pix) ---
model_id = "timbrooks/instruct-pix2pix"

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

# Optional (if using CUDA + enough VRAM): pipe.enable_xformers_memory_efficient_attention()

print('Model loaded.')


Fetching 15 files:   7%|▋         | 1/15 [00:00<00:10,  1.36it/s]

In [ ]:
# --- Diffusion-based rotation attempt ---
def rotation_instruction(angle_deg):
    direction = "counterclockwise" if angle_deg > 0 else "clockwise"
    return f"Rotate the image by {abs(int(angle_deg))} degrees {direction}."

def run_pix2pix(img_np, angle_deg, seed=0, steps=20, guidance_scale=7.5, image_guidance_scale=1.5):
    prompt = rotation_instruction(angle_deg)

    # InstructPix2Pix expects 512x512 images; resize up for the model then back down.
    pil_in = Image.fromarray((img_np * 255).astype(np.uint8)).resize((512, 512), Image.BICUBIC)

    generator = torch.Generator(device=device).manual_seed(seed)

    if device == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            result = pipe(
                prompt=prompt,
                image=pil_in,
                num_inference_steps=steps,
                guidance_scale=guidance_scale,
                image_guidance_scale=image_guidance_scale,
                generator=generator,
            ).images[0]
    else:
        result = pipe(
            prompt=prompt,
            image=pil_in,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            image_guidance_scale=image_guidance_scale,
            generator=generator,
        ).images[0]

    # Resize back to original size for comparison.
    result = result.resize((img_np.shape[1], img_np.shape[0]), Image.BILINEAR)
    out = np.asarray(result).astype(np.float32) / 255.0
    return out


In [ ]:
# --- Visualize results ---
def plot_triplets(samples, title=None):
    n = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = np.array([axes])

    for i, (src, target, pred, angle) in enumerate(samples):
        axes[i, 0].imshow(src)
        axes[i, 0].set_title(f'Input')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(target)
        axes[i, 1].set_title(f'Target rot ({angle} deg)')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(pred)
        axes[i, 2].set_title('Diffusion output')
        axes[i, 2].axis('off')

    if title:
        fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()


In [ ]:
# Run a small probe
angles = [15, 30, 45, -60]
samples = []

base_images = make_sample_batch(n=len(angles), size=128, num_shapes=4, seed=7)

for i, (img, angle) in enumerate(zip(base_images, angles)):
    target = rotate_image_np(img, angle)
    pred = run_pix2pix(img, angle, seed=100 + i, steps=20)
    samples.append((img, target, pred, angle))

plot_triplets(samples, title='Diffusion Rotation Attempt on Colored Shapes')


In [ ]:
# Optional: compute a rough pixel MSE vs. the geometric target (lower is better).
def mse(a, b):
    return float(np.mean((a - b) ** 2))

for i, (src, tgt, pred, angle) in enumerate(samples):
    print(f'Angle {angle:+} deg | MSE: {mse(pred, tgt):.4f}')


## Notes

- InstructPix2Pix is trained for natural images, so clean geometric rotations on synthetic shapes are not guaranteed.
- If results are too noisy, try fewer angles, increase `image_guidance_scale`, or lower `guidance_scale`.
- For more faithful geometric transforms, specialized spatial transformers or diffusion models fine-tuned on the dataset will perform better.
